<a href="https://colab.research.google.com/github/hyeonji-jang65/econ81230-project-jang/blob/main/code/03_descriptives.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03. Descriptive statistics

Two tables by plan type: values at the reference wave (before filling) and after filling from the previous wave.
Means with standard deviations in parentheses; the column next to each mean is the missing rate (%).
Saved as CSV and LaTeX.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np, pandas as pd

OUTDIR = '/content/drive/MyDrive/HRS/'
BASE_YEAR = 2022
pre  = pd.read_parquet(OUTDIR + 'features_prefill.parquet')
post = pd.read_parquet(OUTDIR + 'features.parquet')

K = 1000
# (column, label, kind, scale): kind = cont | bin | cat ; values are divided by scale
SPEC = [
    ('Outcome', None, 'head', 1),
    ('pen_m',         f'Monthly pension ({BASE_YEAR} $)', 'cont', 1),
    ('ln_pen',        'Log monthly pension', 'cont', 1),
    ('Demographics', None, 'head', 1),
    ('female',        'Female', 'bin', 1),
    ('educ_yrs',      'Years of education', 'cont', 1),
    ('race',          'Race/ethnicity', 'cat', 1),
    ('veteran',       'Veteran', 'bin', 1),
    ('birth_year',    'Birth year', 'cont', 1),
    ('Work history (last job before claiming, t1)', None, 'head', 1),
    ('earn_t1',       f'Annual earnings ({BASE_YEAR} $1,000)', 'cont', K),
    ('fas',           f'Final average earnings, t1 and t1-1 ({BASE_YEAR} $1,000)', 'cont', K),
    ('tenure',        'Job tenure (years)', 'cont', 1),
    ('hours',         'Weekly hours', 'cont', 1),
    ('occ',           'Occupation', 'cat', 1),
    ('ind',           'Industry', 'cat', 1),
    ('n_plans',       'Number of pension plans', 'cont', 1),
    ('dc_balance',    f'DC account balance ({BASE_YEAR} $1,000)', 'cont', K),
    ('Claiming conditions (t0)', None, 'head', 1),
    ('claim_age',     'Age at first receipt', 'cont', 1),
    ('claim_age_grp', 'Age group at first receipt', 'cat', 1),
    ('marital',       'Marital status', 'cat', 1),
    ('ss_receipt',    'Receives Social Security', 'bin', 1),
    ('region',        'Census region', 'cat', 1),
    ('gap_waves',     'Waves between t1 and t0', 'cont', 1),
    ('claim_year',    'Interview year at t0', 'cont', 1),
    ('Extended set (t1 unless noted)', None, 'head', 1),
    ('total_wealth',  f'Household total wealth, all assets ({BASE_YEAR} $1,000)', 'cont', K),
    ('fin_wealth',    f'Household financial wealth ({BASE_YEAR} $1,000)', 'cont', K),
    ('ira',           f'IRA balance ({BASE_YEAR} $1,000)', 'cont', K),
    ('home_value',    f'Primary residence value ({BASE_YEAR} $1,000)', 'cont', K),
    ('hh_income',     f'Household income ({BASE_YEAR} $1,000)', 'cont', K),
    ('sp_earn',       f'Spouse earnings ({BASE_YEAR} $1,000)', 'cont', K),
    ('sp_pension',    'Spouse receives a pension (t0)', 'bin', 1),
    ('health',        'Self-rated health (1 = excellent, 5 = poor)', 'cont', 1),
    ('p_live75',      'Subjective probability of living to 75', 'cont', 1),
    ('cognition',     'Cognition score (27-point, mode-adjusted)', 'cont', 1),
]
ORDER = {  # display order of categories
    'race': ['White', 'Black', 'Hispanic', 'Other'],
    'occ': ['Managerial/professional', 'Sales/office', 'Service', 'Blue collar'],
    'ind': ['Goods-producing', 'Trade/transport/utilities', 'Other private services',
            'Education/health/professional', 'Public administration'],
    'claim_age_grp': ['<55', '55-59', '60-61', '62-64', '65+'],
    'marital': ['Married/partnered', 'Separated/divorced', 'Widowed', 'Never married'],
    'region': ['Northeast', 'Midwest', 'South', 'West'],
}

YEARS = {'birth_year', 'claim_year'}                  # no thousands separator

def fmt(x, d, sep=True):
    return '' if pd.isna(x) else (f'{x:,.{d}f}' if sep else f'{x:.{d}f}')

def table(df):
    """Rows: mean, then (SD) underneath; missing-rate column next to each mean."""
    out = []
    groups = {tp: df[df['type'] == tp] for tp in ['DB', 'DC']}
    for col, lab, kind, scale in SPEC:
        if kind == 'head':
            out.append([col, '', '', '', '']); continue
        if col not in df:
            continue
        cells_m, cells_s = [], []
        for tp, g in groups.items():
            na = col == 'dc_balance' and tp == 'DB'
            x = g[col]
            miss = '' if na else fmt(x.isna().mean() * 100, 1)
            if kind == 'cat':
                cells_m += ['', miss]; cells_s += ['', '']
                continue
            v = pd.to_numeric(x, errors='coerce') / scale
            d = 2 if kind == 'bin' or abs(v.mean()) < 10 else 1
            sep = col not in YEARS
            cells_m += ['–' if na else fmt(v.mean(), d, sep), miss]
            cells_s += ['' if na else f'({fmt(v.std(), d, sep)})', '']
        out.append(['  ' + lab] + cells_m)
        if kind != 'cat':
            out.append([''] + cells_s)
        else:
            for c in ORDER.get(col, sorted(df[col].dropna().unique())):
                row = ['      ' + c]
                for tp, g in groups.items():
                    x = g[col].dropna()
                    row += [fmt((x == c).mean(), 2) if len(x) else '', '']
                out.append(row)
    out.append(['Observations', f"{len(groups['DB']):,}", '', f"{len(groups['DC']):,}", ''])
    return pd.DataFrame(out, columns=['Variable', 'DB mean', 'DB missing (%)',
                                      'DC mean', 'DC missing (%)'])

def to_latex(t, caption, label):
    esc = lambda s: (str(s).replace('&', r'\&').replace('%', r'\%').replace('$', r'\$')
                     .replace('<', r'$<$').replace('_', r'\_'))
    lines = [r'\begin{table}[!htbp]\centering', rf'\caption{{{caption}}}\label{{{label}}}',
             r'\begin{tabular}{lcccc}', r'\toprule',
             r' & \multicolumn{2}{c}{DB} & \multicolumn{2}{c}{DC} \\',
             r'\cmidrule(lr){2-3}\cmidrule(lr){4-5}',
             r'Variable & Mean & Missing (\%) & Mean & Missing (\%) \\', r'\midrule']
    for r in t.itertuples(index=False):
        name = esc(r[0])
        if r[0] and not r[0].startswith(' ') and r[0] != 'Observations':
            lines.append(rf'\multicolumn{{5}}{{l}}{{\textit{{{name}}}}} \\')
            continue
        indent = (len(r[0]) - len(r[0].lstrip())) // 2
        name = r'\hspace{' + str(indent * 0.5) + 'em}' + esc(r[0].strip()) if indent else name
        if r[0] == 'Observations':
            lines.append(r'\midrule')
        lines.append(' & '.join([name] + [esc(c) for c in r[1:]]) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

In [ ]:
NOTE = (f'Means with standard deviations in parentheses; categorical variables show shares. '
        f'Dollar amounts in {BASE_YEAR} dollars (CPI-U). Missing (%) is the share of respondents '
        'with no value. Final average earnings average the available values at t1 and t1-1 in both tables.')
tabs = {
    'table_desc_prefill': (table(pre),  'Descriptive statistics, values at the reference wave'),
    'table_desc_filled':  (table(post), 'Descriptive statistics, after filling from the previous wave'),
}
for name, (t, cap) in tabs.items():
    print(f'\n=== {cap} ===')
    print(t.to_string(index=False))
    t.to_csv(OUTDIR + name + '.csv', index=False)
    with open(OUTDIR + name + '.tex', 'w') as f:
        f.write(to_latex(t, cap, 'tab:' + name) + '\n% ' + NOTE + '\n')
print('\n' + NOTE)
print('Saved CSV and LaTeX tables to', OUTDIR)